 Creating the PartC/D Notebook 

### Part C,D - Support Copilot

In [1]:

import sys
from pathlib import Path
import pandas as pd

# Locate the project folder automatically
project_folder = Path.cwd()

while not (project_folder / "mock_llm.py").exists():
    if project_folder.parent == project_folder:
        raise FileNotFoundError("Project folder not found.")
    project_folder = project_folder.parent

if str(project_folder) not in sys.path:
    sys.path.insert(0, str(project_folder))

import mock_llm
from mock_llm import est_tokens, chat, reset, TransientLLMError

print("Project folder:", project_folder)
print("mock_llm.py imported successfully.")


Project folder: C:\Users\Lenovo\Desktop\Capstone_Project
mock_llm.py imported successfully.



### Task 1: Input Guardrails

- Skip tickets with empty or whitespace-only text.
- Flag tickets with more than 400 estimated tokens.
- Keep the remaining tickets ready for processing.


In [2]:
import sys
from pathlib import Path
import pandas as pd

# Locate the main project folder automatically
project_folder = Path.cwd().resolve()

while not (project_folder / "mock_llm.py").exists():
    if project_folder.parent == project_folder:
        raise FileNotFoundError("Project folder containing mock_llm.py was not found.")
    project_folder = project_folder.parent

# Add the project folder to Python's import path
if str(project_folder) not in sys.path:
    sys.path.insert(0, str(project_folder))

# Check that the required file exists
mock_llm_path = project_folder / "mock_llm.py"

assert mock_llm_path.exists(), "mock_llm.py was not found in the project folder."

# Import the token estimation function
from mock_llm import est_tokens

print("Project folder:", project_folder)
print("mock_llm.py found successfully.")
print("Token estimation function imported successfully.")

Project folder: C:\Users\Lenovo\Desktop\Capstone_Project
mock_llm.py found successfully.
Token estimation function imported successfully.


 ### Redact Personal Information

In [3]:
tickets = pd.read_csv(
    project_folder / "data" / "raw" / "tickets.csv",
    keep_default_na=False
)

print("Tickets loaded:", len(tickets))
print("Columns:", tickets.columns.tolist())

Tickets loaded: 164
Columns: ['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']


In [4]:
import re

# Required patterns and replacement placeholders
email_pattern = r"[\w.+-]+@[\w-]+\.[\w.]+"
phone_pattern = r"\b[6-9]\d{9}\b"
order_pattern = r"ORD-\d{6}"

redacted_texts = []
email_count = 0
phone_count = 0
order_count = 0

# Redact each ticket in the required order
for text in tickets["text"].astype(str):

    text, email_subs = re.subn(email_pattern, "[EMAIL]", text)
    text, phone_subs = re.subn(phone_pattern, "[PHONE]", text)
    text, order_subs = re.subn(order_pattern, "[ORDER_ID]", text)

    email_count += email_subs
    phone_count += phone_subs
    order_count += order_subs

    redacted_texts.append(text)

# Store redacted text separately
tickets["redacted_text"] = redacted_texts

# Verify that no original personal information remains
assert not tickets["redacted_text"].str.contains(
    email_pattern, regex=True
).any()

assert not tickets["redacted_text"].str.contains(
    phone_pattern, regex=True
).any()

assert not tickets["redacted_text"].str.contains(
    order_pattern, regex=True
).any()

# Display substitution counts
redaction_summary = pd.DataFrame({
    "Information": ["Email", "Phone Number", "Order ID", "Total"],
    "Replacements": [
        email_count,
        phone_count,
        order_count,
        email_count + phone_count + order_count
    ]
})

display(redaction_summary)

print("Remaining personal information: 0")
print("Redaction verification completed successfully.")

,Information,Replacements
0,Email,55
1,Phone Number,91
2,Order ID,132
3,Total,278


Remaining personal information: 0
Redaction verification completed successfully.


I checked a few ticket descriptions after redaction to make sure personal details were replaced with the required placeholders. The original ticket text remains unchanged.

In [5]:
# Check a few tickets before and after redaction
display(
    tickets[["ticket_id", "text", "redacted_text"]].head(5)
)

print("Email replacements:", email_count)
print("Phone replacements:", phone_count)
print("Order ID replacements:", order_count)
print("Total replacements:", email_count + phone_count + order_count)

print("Redaction check completed.")

,ticket_id,text,redacted_text
0,T0024,There is an unauthorized transaction on my car...,There is an unauthorized transaction on my car...
1,T0073,The invoice for ORD-100816 shows a wrong amoun...,The invoice for [ORDER_ID] shows a wrong amoun...
2,T0095,The payment page shows an error and the app fr...,The payment page shows an error and the app fr...
3,T0159,,
4,T0080,I was charged for order ORD-109516 that I neve...,I was charged for order [ORDER_ID] that I neve...


Email replacements: 55
Phone replacements: 91
Order ID replacements: 132
Total replacements: 278
Redaction check completed.


### Task 3: Zero-shot and Few-shot Prompts

created two prompts for ticket classification. 
The zero-shot prompt uses instructions only, while the few-shot prompt includes examples to guide the classification.

In [6]:
# Allowed ticket categories
categories = ["billing", "delivery", "technical", "fraud"]

# Prompt with instructions only
zero_shot_prompt = """
You are a support ticket classification assistant.

CLASSIFY each support ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return only valid JSON in this format:
{"category": "...", "reason": "..."}
Do not include any extra text.
"""

# Prompt with instructions and examples
few_shot_prompt = """
You are a support ticket classification assistant.

CLASSIFY each support ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return only valid JSON in this format:
{"category": "...", "reason": "..."}
Do not include any extra text.

EXAMPLES:

Ticket: I was charged twice for the same order.
Reply: {"category": "billing", "reason": "Duplicate payment"}

Ticket: My order has not arrived even though it was expected yesterday.
Reply: {"category": "delivery", "reason": "Order delivery delay"}

Ticket: Someone used my account to place an order without permission.
Reply: {"category": "fraud", "reason": "Unauthorised account activity"}
"""

# Check the required prompt conditions
assert "CLASSIFY" in zero_shot_prompt
assert "EXAMPLES:" not in zero_shot_prompt

assert "CLASSIFY" in few_shot_prompt
assert "EXAMPLES:" in few_shot_prompt

for category in categories:
    assert category in zero_shot_prompt
    assert category in few_shot_prompt

assert '"category": "..."' in zero_shot_prompt
assert '"reason": "..."' in zero_shot_prompt
assert '"category": "..."' in few_shot_prompt
assert '"reason": "..."' in few_shot_prompt

print("Zero-shot and few-shot prompts created successfully.")
print("All required categories and output format checks passed.")

Zero-shot and few-shot prompts created successfully.
All required categories and output format checks passed.


### Task 4: Handle Temporary Failures

This helps process the tickets without stopping the entire workflow because of temporary errors.

In [7]:
import random
import time
from mock_llm import TransientLLMError, chat, reset

BASE_DELAY = 0.05
MAX_RETRIES = 3

def call_with_retry(messages, ticket_id):
    for retry_no in range(MAX_RETRIES + 1):
        try:
            return chat(messages)

        except TransientLLMError:
            if retry_no == MAX_RETRIES:
                raise

            wait_time = (
                BASE_DELAY
                * (2 ** retry_no)
                * random.uniform(0.9, 1.15)
            )

            print(
                f"Ticket {ticket_id} | "
                f"Retry {retry_no} | "
                f"Waiting {wait_time:.4f} seconds"
            )

            time.sleep(wait_time)

In [8]:
print("Retry function created successfully.")
print("Maximum retries per ticket:", MAX_RETRIES)
print("Base delay:", BASE_DELAY, "seconds")

assert callable(call_with_retry)
print("Retry function verification completed.")

Retry function created successfully.
Maximum retries per ticket: 3
Base delay: 0.05 seconds
Retry function verification completed.


### Validate Model Responses

I checked the model responses and accepted only valid categories, flagging unclear or invalid answers for human review.

In [9]:
import json
import re

ALLOWED_CATEGORIES = {"billing", "delivery", "technical", "fraud"}

def validate_response(response_text):
    try:
        result = json.loads(response_text)

        if isinstance(result, dict):
            category = result.get("category")

            if isinstance(category, str) and category.lower() in ALLOWED_CATEGORIES:
                return category.lower(), "classified_json"

        return None, "flagged_invalid_output"

    except (json.JSONDecodeError, TypeError):
        words = re.findall(r"[a-z]+", str(response_text).lower())
        matches = ALLOWED_CATEGORIES.intersection(words)

        if len(matches) == 1:
            return matches.pop(), "classified_keyword"

        return None, "flagged_invalid_output"


# Check the response validation rules
test_responses = [
    '{"category": "billing", "reason": "Payment issue"}',
    "This appears to be billing-related.",
    "Unable to identify the correct category."
]

for response in test_responses:
    category, status = validate_response(response)
    print(f"Category: {category} | Status: {status}")

Category: billing | Status: classified_json
Category: billing | Status: classified_keyword
Category: None | Status: flagged_invalid_output


In [10]:
import inspect
from mock_llm import chat

print("Chat function:", inspect.signature(chat))

Chat function: (messages, tools=None)


In [11]:
import inspect
print(inspect.getsource(chat))

def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



In [12]:
import mock_llm
print(inspect.getsource(mock_llm._classify))

def _classify(messages):
    system, user = messages[0]["content"], messages[-1]["content"]
    _state["calls"] += 1
    if _state["calls"] % 9 == 0:                                 # every 9th call fails once
        raise TransientLLMError("503 server busy")
    _state["ok"] += 1
    order = ["fraud", "technical", "billing", "delivery"] if "EXAMPLES:" in system else ["billing", "fraud", "technical", "delivery"]
    w = _words(user)
    cat = next((c for c in order if KW[c] & w), "technical")
    if _state["ok"] % 13 == 0:                                   # chatty, non-JSON answer
        return _resp(f"The customer appears to have a {cat}-related problem.", messages=messages)
    if _state["ok"] % 17 == 0:                                   # JSON, but category outside the allowed list
        return _resp(json.dumps({"category": "shipping", "reason": "parcel movement"}), messages=messages)
    return _resp(json.dumps({"category": cat, "reason": "keyword match"}), messages=messages)


In [13]:
print(inspect.getsource(mock_llm._resp))

def _resp(content=None, tool_calls=None, finish="stop", messages=()):
    prompt = sum(est_tokens(m.get("content") or json.dumps(m.get("tool_calls", ""), default=str)) for m in messages)
    comp = est_tokens(content or json.dumps(tool_calls or "", default=str))
    msg = NS(role="assistant", content=content, tool_calls=[NS(**tc) for tc in (tool_calls or [])])
    return NS(choices=[NS(message=msg, finish_reason=finish)],
              usage=NS(prompt_tokens=prompt, completion_tokens=comp, total_tokens=prompt + comp))



### Task 5 & 6: Response Validation and Prompt Comparison

I checked the model responses and compared both prompts using accuracy and token usage.

In [14]:

import sys
import json
import re
from pathlib import Path

import pandas as pd
import mock_llm
from mock_llm import est_tokens


# --------------------------------------------------
# 1. Locate project folder and load ticket data
# --------------------------------------------------

ROOT = Path.cwd().resolve()

while not (ROOT / "generate_data.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project folder was not found.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

tickets = pd.read_csv(ROOT / "data" / "raw" / "tickets.csv")

required_columns = {
    "ticket_id", "text", "true_category"
}

missing_columns = required_columns - set(tickets.columns)

if missing_columns:
    raise ValueError(f"Missing ticket columns: {missing_columns}")


# --------------------------------------------------
# 2. Prepare redacted ticket text
# --------------------------------------------------

email_pattern = r"[\w.+-]+@[\w-]+\.[\w.]+"
phone_pattern = r"\b[6-9]\d{9}\b"
order_pattern = r"ORD-\d{6}"

def redact_ticket(text):
    text = str(text)
    text = re.sub(email_pattern, "[EMAIL]", text)
    text = re.sub(phone_pattern, "[PHONE]", text)
    text = re.sub(order_pattern, "[ORDER_ID]", text)
    return text


tickets["redacted_text"] = tickets["text"].apply(redact_ticket)

# Confirm that no personal information remains
for pattern in [email_pattern, phone_pattern, order_pattern]:
    assert not tickets["redacted_text"].str.contains(
        pattern, regex=True
    ).any(), "Unredacted personal information found."


# --------------------------------------------------
# 3. Define zero-shot and few-shot prompts
# --------------------------------------------------

ALLOWED_CATEGORIES = {
    "billing", "delivery", "technical", "fraud"
}

zero_shot_prompt = """
You are a customer support ticket classification assistant.

CLASSIFY each ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return JSON only in this format:
{"category": "...", "reason": "..."}

The category must be one of the four allowed categories.
Do not include text outside the JSON response.
"""

few_shot_prompt = """
You are a customer support ticket classification assistant.

CLASSIFY each ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return JSON only in this format:
{"category": "...", "reason": "..."}

The category must be one of the four allowed categories.
Do not include text outside the JSON response.

EXAMPLES:

Ticket: I was charged twice for my order.
Response: {"category": "billing", "reason": "Duplicate charge"}

Ticket: My order has not arrived yet.
Response: {"category": "delivery", "reason": "Order delivery delay"}

Ticket: The application keeps showing an error.
Response: {"category": "technical", "reason": "Application error"}

Ticket: Someone used my account without permission.
Response: {"category": "fraud", "reason": "Unauthorized account activity"}
"""


assert "CLASSIFY" in zero_shot_prompt
assert "EXAMPLES:" not in zero_shot_prompt
assert "CLASSIFY" in few_shot_prompt
assert "EXAMPLES:" in few_shot_prompt


# --------------------------------------------------
# 4. Validate model responses
# --------------------------------------------------

def validate_response(response_text):

    try:
        result = json.loads(response_text)

        if isinstance(result, dict):
            category = result.get("category")

            if (
                isinstance(category, str)
                and category.lower() in ALLOWED_CATEGORIES
            ):
                return category.lower(), "classified_json"

        return None, "flagged_invalid_output"

    except (json.JSONDecodeError, TypeError):

        words = re.findall(
            r"[a-z]+", str(response_text).lower()
        )

        matches = ALLOWED_CATEGORIES.intersection(words)

        if len(matches) == 1:
            return matches.pop(), "classified_keyword"

        return None, "flagged_invalid_output"


# --------------------------------------------------
# 5. Run classification for each prompt
# --------------------------------------------------

MAX_INPUT_TOKENS = 400

if "call_with_retry" not in globals():
    raise RuntimeError(
        "Run the Task 4 retry-function cell before this cell."
    )


def run_classification(prompt, prompt_name):

    mock_llm.reset()
    results = []

    for _, row in tickets.iterrows():

        text = str(row["text"])
        redacted_text = row["redacted_text"]

        if not text.strip():

            status = "skipped_empty"
            category = None
            tokens = 0

        elif est_tokens(text) > MAX_INPUT_TOKENS:

            status = "flagged_long"
            category = None
            tokens = 0

        else:

            messages = [
                {"role": "system", "content": prompt},
                {"role": "user", "content": redacted_text}
            ]

            # Privacy checks before model call
            assert not re.search(email_pattern, redacted_text)
            assert not re.search(phone_pattern, redacted_text)
            assert not re.search(order_pattern, redacted_text)

            response = call_with_retry(
                messages, row["ticket_id"]
            )

            response_text = response.choices[0].message.content

            category, status = validate_response(response_text)

            tokens = response.usage.total_tokens

        results.append({
            "ticket_id": row["ticket_id"],
            "true_category": row["true_category"],
            "category": category,
            "status": status,
            "tokens_used": tokens
        })

    result_df = pd.DataFrame(results)

    classified = result_df[
        result_df["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ]

    accuracy = (
        classified["category"].eq(
            classified["true_category"]
        ).mean() * 100
        if len(classified) > 0 else 0
    )

    total_tokens = int(result_df["tokens_used"].sum())

    summary = {
        "Prompt": prompt_name,
        "Tickets classified": len(classified),
        "Accuracy (%)": round(accuracy, 2),
        "Total tokens": total_tokens,
        "Tokens per classified ticket": round(
            total_tokens / len(classified), 2
        ) if len(classified) > 0 else 0
    }

    print(f"\n{prompt_name} status counts:")
    print(result_df["status"].value_counts())

    return result_df, summary


# --------------------------------------------------
# 6. Compare zero-shot and few-shot
# --------------------------------------------------

zero_shot_results, zero_summary = run_classification(
    zero_shot_prompt, "Zero-shot"
)

few_shot_results, few_summary = run_classification(
    few_shot_prompt, "Few-shot"
)

comparison = pd.DataFrame([
    zero_summary,
    few_summary
])

print("\nPrompt comparison:")
display(comparison)


# --------------------------------------------------
# 7. Zero-shot confusion table
# --------------------------------------------------

zero_classified = zero_shot_results[
    zero_shot_results["status"].isin(
        ["classified_json", "classified_keyword"]
    )
]

confusion_table = pd.crosstab(
    zero_classified["true_category"],
    zero_classified["category"]
).reindex(
    index=sorted(ALLOWED_CATEGORIES),
    columns=sorted(ALLOWED_CATEGORIES),
    fill_value=0
)

print("\nZero-shot confusion table:")
display(confusion_table)


# --------------------------------------------------
# 8. Final comparison
# --------------------------------------------------

print("\nPrompt comparison completed.")


Ticket T0037 | Retry 0 | Waiting 0.0522 seconds


Ticket T0123 | Retry 0 | Waiting 0.0476 seconds

Ticket T0049 | Retry 0 | Waiting 0.0481 seconds
Ticket T0134 | Retry 0 | Waiting 0.0548 seconds


Ticket T0137 | Retry 0 | Waiting 0.0477 seconds


Ticket T0060 | Retry 0 | Waiting 0.0518 seconds


Ticket T0017 | Retry 0 | Waiting 0.0461 seconds


... (17 outputs left out)

,Prompt,Tickets classified,Accuracy (%),Total tokens,Tokens per classified ticket
0,Zero-shot,151,82.12,14684,97.25
1,Few-shot,151,97.35,26364,174.60



Zero-shot confusion table:


category,billing,delivery,fraud,technical
true_category,,,,
billing,38,0,0,0
delivery,0,38,0,0
fraud,13,0,22,0
technical,10,0,0,26



Prompt comparison completed.


### Task 7: Token Budget and Final Ticket Routing

In [15]:

import re
import sys
from pathlib import Path
import pandas as pd
import mock_llm
from mock_llm import est_tokens

# Locate project folder
ROOT = Path.cwd().resolve()

while not (ROOT / "generate_data.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project folder not found.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Load ticket and customer segment data
tickets = pd.read_csv(ROOT / "data" / "raw" / "tickets.csv")
segments = pd.read_csv(OUTPUT_DIR / "customer_segments.csv")

# PII redaction
def redact_ticket(text):
    text = str(text)
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", text)
    text = re.sub(r"\b[6-9]\d{9}\b", "[PHONE]", text)
    text = re.sub(r"ORD-\d{6}", "[ORDER_ID]", text)
    return text

tickets["redacted_text"] = tickets["text"].apply(
    lambda x: redact_ticket(x) if pd.notna(x) else ""
)

# Run pipeline with token budget
def run_budget_pipeline(prompt, token_budget):
    mock_llm.reset()

    results = []
    total_tokens = 0

    for _, row in tickets.iterrows():

        ticket_id = row["ticket_id"]
        raw_text = row["text"]

        category = None
        tokens_used = 0

        if pd.isna(raw_text) or not str(raw_text).strip():
            status = "skipped_empty"

        else:
            text = str(raw_text)

            if est_tokens(text) > MAX_INPUT_TOKENS:
                status = "flagged_long"

            elif total_tokens >= token_budget:
                status = "budget_exhausted"

            else:
                messages = [
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": row["redacted_text"]}
                ]

                response = call_with_retry(messages, ticket_id)

                response_text = response.choices[0].message.content
                category, status = validate_response(response_text)

                tokens_used = response.usage.total_tokens
                total_tokens += tokens_used

        results.append({
            "ticket_id": ticket_id,
            "customer_id": row["customer_id"],
            "status": status,
            "category": category,
            "tokens_used": tokens_used
        })

    return pd.DataFrame(results), total_tokens


# Zero-shot budget demonstration
TOKEN_BUDGET = 6000

budget_results, budget_total = run_budget_pipeline(
    zero_shot_prompt,
    TOKEN_BUDGET
)

budget_path = OUTPUT_DIR / "triage_budget_demo.csv"
budget_results.to_csv(budget_path, index=False)

print("Zero-shot budget demonstration")
print("Token budget:", TOKEN_BUDGET)
print("Tokens used:", budget_total)
print("\nStatus counts:")
print(budget_results["status"].value_counts())
print("\nSaved:", budget_path)


# Final few-shot pipeline
TOKEN_BUDGET = 100000

final_results, final_total = run_budget_pipeline(
    few_shot_prompt,
    TOKEN_BUDGET
)

# Add customer segment information
final_results = final_results.merge(
    segments[["customer_id", "segment_name", "is_premium"]],
    on="customer_id",
    how="left"
)

# Final routing rules
def assign_route(row):

    if row["status"] in [
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ]:
        return "human_review", "NORMAL"

    if row["category"] == "fraud":
        return "security_team", "HIGH"

    if row["is_premium"] == True:
        return "premium_desk", "HIGH"

    return "support_queue", "NORMAL"


final_results[["route", "priority"]] = final_results.apply(
    lambda row: pd.Series(assign_route(row)),
    axis=1
)

# Keep required output columns
final_results = final_results[
    [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used"
    ]
]

# Save final output
final_path = OUTPUT_DIR / "triage_results.csv"
final_results.to_csv(final_path, index=False)

print("\nFinal few-shot pipeline")
print("Token budget:", TOKEN_BUDGET)
print("Tokens used:", final_total)

print("\nStatus counts:")
print(final_results["status"].value_counts())

print("\nRouting counts:")
print(final_results["route"].value_counts())

print("\nFinal output shape:", final_results.shape)
print("Saved:", final_path)


Ticket T0002 | Retry 0 | Waiting 0.0451 seconds
Ticket T0116 | Retry 0 | Waiting 0.0457 seconds


Ticket T0153 | Retry 0 | Waiting 0.0485 seconds


Ticket T0039 | Retry 0 | Waiting 0.0527 seconds


Ticket T0152 | Retry 0 | Waiting 0.0457 seconds
Ticket T0043 | Retry 0 | Waiting 0.0490 seconds


Ticket T0111 | Retry 0 | Waiting 0.0517 seconds
Ticket T0155 | Retry 0 | Waiting 0.0565 seconds


Zero-shot budget demonstration
Token budget: 6000
Tokens used: 6088

Status counts:
status
budget_exhausted          90
classified_json           58
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     3
Name: count, dtype: int64

Saved: C:\Users\Lenovo\Desktop\Capstone_Project\outputs\triage_budget_demo.csv
Ticket T0002 | Retry 0 | Waiting 0.0476 seconds
Ticket T0116 | Retry 0 | Waiting 0.0539 seconds
Ticket T0153 | Retry 0 | Waiting 0.0470 seconds
Ticket T0039 | Retry 0 | Waiting 0.0533 seconds


Ticket T0152 | Retry 0 | Waiting 0.0554 seconds
Ticket T0043 | Retry 0 | Waiting 0.0557 seconds
Ticket T0111 | Retry 0 | Waiting 0.0542 seconds
Ticket T0155 | Retry 0 | Waiting 0.0537 seconds


Ticket T0100 | Retry 0 | Waiting 0.0479 seconds
Ticket T0068 | Retry 0 | Waiting 0.0521 seconds
Ticket T0135 | Retry 0 | Waiting 0.0538 seconds
Ticket T0009 | Retry 0 | Waiting 0.0534 seconds


Ticket T0067 | Retry 0 | Waiting 0.0530 seconds
Ticket T0106 | Retry 0 | Waiting 0.0476 seconds
Ticket T0097 | Retry 0 | Waiting 0.0489 seconds
Ticket T0003 | Retry 0 | Waiting 0.0528 seconds


Ticket T0019 | Retry 0 | Waiting 0.0484 seconds
Ticket T0086 | Retry 0 | Waiting 0.0553 seconds
Ticket T0065 | Retry 0 | Waiting 0.0530 seconds



Final few-shot pipeline
Token budget: 100000
Tokens used: 25780

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Routing counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Final output shape: (164, 8)
Saved: C:\Users\Lenovo\Desktop\Capstone_Project\outputs\triage_results.csv


## Task 7: Budget, Routing and Output

A token budget is applied to control model usage. 
Tickets that exceed the available budget are sent for human review.

In [16]:
import pandas as pd
import re
from pathlib import Path

# Project paths
DATA_PATH = ROOT / "data" / "raw" / "tickets.csv"
SEGMENT_PATH = ROOT / "outputs" / "customer_segments.csv"
OUTPUT_PATH = ROOT / "outputs"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load tickets without converting empty strings to NaN
tickets = pd.read_csv(DATA_PATH, keep_default_na=False)

# Load customer segments from Part B
segments = pd.read_csv(SEGMENT_PATH)


# --------------------------------------------------
# Budget-controlled classification pipeline
# --------------------------------------------------

def run_budget_pipeline(prompt, prompt_name, TOKEN_BUDGET):

    mock_llm.reset()

    total_tokens = 0
    results = []

    for _, row in tickets.iterrows():

        ticket_id = row["ticket_id"]
        customer_id = row["customer_id"]
        raw_text = row["text"]

        category = None
        status = None
        ticket_tokens = 0

        # 1. Skip empty tickets
        if pd.isna(raw_text) or not str(raw_text).strip():

            status = "skipped_empty"

        # 2. Flag unusually long tickets
        elif est_tokens(str(raw_text)) > MAX_INPUT_TOKENS:

            status = "flagged_long"

        else:

            # 3. Redact personal information
            redacted_text = str(raw_text)

            redacted_text = re.sub(
                r"[\w.+-]+@[\w-]+\.[\w.]+",
                "[EMAIL]",
                redacted_text
            )

            redacted_text = re.sub(
                r"\b[6-9]\d{9}\b",
                "[PHONE]",
                redacted_text
            )

            redacted_text = re.sub(
                r"ORD-\d{6}",
                "[ORDER_ID]",
                redacted_text
            )

            # Verify that no personal information remains
            assert not re.search(
                r"[\w.+-]+@[\w-]+\.[\w.]+",
                redacted_text
            )

            assert not re.search(
                r"\b[6-9]\d{9}\b",
                redacted_text
            )

            assert not re.search(
                r"ORD-\d{6}",
                redacted_text
            )

            # 4. Check token budget before model call
            if total_tokens >= TOKEN_BUDGET:

                status = "budget_exhausted"

            else:

                messages = [
                    {
                        "role": "system",
                        "content": prompt
                    },
                    {
                        "role": "user",
                        "content": redacted_text
                    }
                ]

                # 5. Call model with retry handling
                response = call_with_retry(
                    messages,
                    ticket_id
                )

                ticket_tokens = response.usage.total_tokens
                total_tokens += ticket_tokens

                # 6. Validate model response
                response_text = response.choices[0].message.content

                category, status = validate_response(
                    response_text
                )

        results.append({
            "ticket_id": ticket_id,
            "customer_id": customer_id,
            "status": status,
            "category": category,
            "tokens_used": ticket_tokens
        })

    result_df = pd.DataFrame(results)

    print(f"\n{prompt_name} Budget Summary")
    print("-" * 45)
    print("Token budget:", TOKEN_BUDGET)
    print("Total tokens used:", total_tokens)
    print("\nStatus counts:")
    print(result_df["status"].value_counts())

    return result_df, total_tokens


# --------------------------------------------------
# Part A: Zero-shot budget demonstration
# --------------------------------------------------

budget_demo, budget_tokens = run_budget_pipeline(
    zero_shot_prompt,
    "Zero-shot",
    TOKEN_BUDGET=6000
)

budget_demo.to_csv(
    OUTPUT_PATH / "triage_budget_demo.csv",
    index=False
)

print("\nBudget demonstration saved successfully.")


# --------------------------------------------------
# Part B: Few-shot final classification
# --------------------------------------------------

final_results, final_tokens = run_budget_pipeline(
    few_shot_prompt,
    "Few-shot Final",
    TOKEN_BUDGET=100000
)


# --------------------------------------------------
# Part C: Merge customer segments
# --------------------------------------------------

final_results = final_results.merge(
    segments[["customer_id", "segment_name", "is_premium"]],
    on="customer_id",
    how="left"
)


# --------------------------------------------------
# Part D: Ticket routing and priority
# --------------------------------------------------

def assign_routing(row):

    if row["status"] in [
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ]:
        return pd.Series(["NORMAL", "human_review"])

    elif row["category"] == "fraud":
        return pd.Series(["HIGH", "security_team"])

    elif row["is_premium"] == True:
        return pd.Series(["HIGH", "premium_desk"])

    else:
        return pd.Series(["NORMAL", "support_queue"])


final_results[["priority", "route"]] = final_results.apply(
    assign_routing,
    axis=1
)


# --------------------------------------------------
# Final output
# --------------------------------------------------

final_results = final_results[
    [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used"
    ]
]

# Save final results
final_results.to_csv(
    OUTPUT_PATH / "triage_results.csv",
    index=False
)


# --------------------------------------------------
# Verification
# --------------------------------------------------

print("\nFinal Triage Results")
print("=" * 45)

print("Total tickets:", len(final_results))

print("\nStatus counts:")
print(final_results["status"].value_counts())

print("\nRouting counts:")
print(final_results["route"].value_counts())

print("\nPriority counts:")
print(final_results["priority"].value_counts())

print("\nOutput shape:", final_results.shape)

print("\nFirst five records:")
display(final_results.head())

print("\nFinal output saved successfully.")

Ticket T0002 | Retry 0 | Waiting 0.0535 seconds


Ticket T0116 | Retry 0 | Waiting 0.0498 seconds
Ticket T0153 | Retry 0 | Waiting 0.0544 seconds
Ticket T0039 | Retry 0 | Waiting 0.0550 seconds


Ticket T0152 | Retry 0 | Waiting 0.0527 seconds


Ticket T0043 | Retry 0 | Waiting 0.0460 seconds
Ticket T0111 | Retry 0 | Waiting 0.0523 seconds
Ticket T0155 | Retry 0 | Waiting 0.0491 seconds



Zero-shot Budget Summary
---------------------------------------------
Token budget: 6000
Total tokens used: 6088

Status counts:
status
budget_exhausted          90
classified_json           58
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     3
Name: count, dtype: int64

Budget demonstration saved successfully.
Ticket T0002 | Retry 0 | Waiting 0.0553 seconds
Ticket T0116 | Retry 0 | Waiting 0.0522 seconds
Ticket T0153 | Retry 0 | Waiting 0.0485 seconds
Ticket T0039 | Retry 0 | Waiting 0.0520 seconds


Ticket T0152 | Retry 0 | Waiting 0.0464 seconds
Ticket T0043 | Retry 0 | Waiting 0.0551 seconds
Ticket T0111 | Retry 0 | Waiting 0.0496 seconds
Ticket T0155 | Retry 0 | Waiting 0.0559 seconds


... (2 outputs left out)

Ticket T0019 | Retry 0 | Waiting 0.0535 seconds
Ticket T0086 | Retry 0 | Waiting 0.0553 seconds
Ticket T0065 | Retry 0 | Waiting 0.0467 seconds

Few-shot Final Budget Summary
---------------------------------------------
Token budget: 100000
Total tokens used: 25780

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64



Final Triage Results
Total tickets: 164

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Routing counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Priority counts:
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64

Output shape: (164, 8)

First five records:


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At-Risk Customers,HIGH,security_team,169
1,T0073,C0080,classified_json,billing,Loyal Customers,NORMAL,support_queue,165
2,T0095,C1087,classified_json,technical,At-Risk Customers,NORMAL,support_queue,167
3,T0159,C1172,skipped_empty,NaN,Loyal Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At-Risk Customers,HIGH,security_team,169



Final output saved successfully.


### Part D

### Read and split the five policy documents

In [47]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

POLICY_DIR = PROJECT_ROOT / "data" / "policies"

print("Project root:", PROJECT_ROOT)
print("Policy folder exists:", POLICY_DIR.exists())

Project root: c:\Users\Lenovo\Desktop\Capstone_Project
Policy folder exists: True


In [51]:
from pathlib import Path

# Locate the project folder automatically
PROJECT_ROOT = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "data" / "policies").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder containing data/policies."
    )

# Locate the policy documents
POLICY_DIR = PROJECT_ROOT / "data" / "policies"
policy_files = sorted(POLICY_DIR.glob("*.txt"))

if len(policy_files) != 5:
    raise FileNotFoundError(
        f"Expected 5 policy documents, but found {len(policy_files)} "
        f"in {POLICY_DIR}"
    )

# Split each policy into 40-word chunks with 10-word overlap
chunks = []

for file_path in policy_files:
    words = file_path.read_text(encoding="utf-8").split()

    start = 0

    while start < len(words):
        end = start + 40

        chunks.append({
            "source": file_path.name,
            "text": " ".join(words[start:end])
        })

        if end >= len(words):
            break

        start = end - 10

# Display the chunking results
print("Project folder:", PROJECT_ROOT)
print("Policy documents:", len(policy_files))
print("Total chunks:", len(chunks))

for file_path in policy_files:
    count = sum(
        chunk["source"] == file_path.name
        for chunk in chunks
    )
    print(f"{file_path.name}: {count}")

Project folder: c:\Users\Lenovo\Desktop\Capstone_Project
Policy documents: 5
Total chunks: 15
delivery_policy.txt: 3
escalation_policy.txt: 3
payment_policy.txt: 3
privacy_policy.txt: 2
refund_policy.txt: 4


In [54]:
import sys
import inspect

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mock_llm

print("Mock LLM file:", mock_llm.__file__)
print("Chat function:", inspect.signature(mock_llm.chat))
print(inspect.getsource(mock_llm.chat))

Mock LLM file: c:\Users\Lenovo\Desktop\Capstone_Project\mock_llm.py
Chat function: (messages, tools=None)
def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



contect a

In [55]:
def answer(question):

    # Retrieve the three most relevant policy chunks
    retrieved_chunks = retrieve(question, k=3)

    # Prepare the retrieved policy information as context
    context = ""

    for result in retrieved_chunks:
        context += (
            f"[{result['source']}]\n"
            f"{result['text']}\n"
        )

    # Prepare the messages for the mock LLM
    messages = [
        {
            "role": "system",
            "content": (
                "You are a POLICY ASSISTANT. "
                "Answer the question using only the supplied policy context. "
                "Do not make up or assume information. "
                "If the answer is not available in the context, "
                "respond exactly with: "
                "'I could not find this information in the available policies.'"
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n"
                f"{context}"
                f"QUESTION:\n"
                f"{question}"
            )
        }
    ]

    # Generate the policy-based answer
    response = mock_llm.chat(messages)

    # Return the answer and the policy sources used
    sources_used = list(dict.fromkeys(
        result["source"] for result in retrieved_chunks
    ))

    return {
        "answer": response,
        "sources": sources_used
    }

### TF-IDF Vectorisation and Policy Retrieval

converting the policy chunks into TF-IDF vectors so that relevant information can be retrieved based on a user's question. using the  Cosine similarity to rank the chunks.

In [57]:
def retrieve(query, k=3):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks[index]["source"],
            "score": float(scores[index]),
            "text": chunks[index]["text"]
        })

    return results

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Policy chunks:", tfidf_matrix.shape[0])
print("TF-IDF features:", tfidf_matrix.shape[1])

Policy chunks: 15
TF-IDF features: 200


### Retrieval function

In [ ]:
def retrieve(query, k=3):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks[index]["source"],
            "score": float(scores[index]),
            "text": chunks[index]["text"]
        })

    return results

### Retrieval test

In [59]:
def retrieve(query, k=3):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks[index]["source"],
            "score": float(scores[index]),
            "text": chunks[index]["text"]
        })

    return results

### Context Augmentation and Answer Generation

In [61]:
import sys
import mock_llm

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Mock LLM loaded from:", mock_llm.__file__)

Mock LLM loaded from: c:\Users\Lenovo\Desktop\Capstone_Project\mock_llm.py


In [60]:
def answer(question):

    retrieved_chunks = retrieve(question, k=3)

    context = ""

    for result in retrieved_chunks:
        context += (
            f"[{result['source']}]\n"
            f"{result['text']}\n"
        )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a POLICY ASSISTANT. "
                "Answer only using the supplied policy context. "
                "Do not invent or assume information. "
                "If the answer is not available, respond with: "
                "I could not find this information in the available policies."
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}"
                f"QUESTION:\n{question}"
            )
        }
    ]

    response = mock_llm.chat(messages)

    sources_used = list(dict.fromkeys(
        result["source"] for result in retrieved_chunks
    ))

    return {
        "answer": response,
        "sources": sources_used
    }

In [62]:
question = "What is the refund policy for late delivery?"

result = answer(question)

print("Question:", question)
print("\nAnswer:")
print(result["answer"])

print("\nSources used:")
for source in result["sources"]:
    print("-", source)

Question: What is the refund policy for late delivery?

Answer:
namespace(choices=[namespace(message=namespace(role='assistant', content='Refunds up to INR 1000 are\n \nRefund Policy.', tool_calls=[]), finish_reason='stop')], usage=namespace(prompt_tokens=220, completion_tokens=11, total_tokens=231))

Sources used:
- refund_policy.txt


### Retrieval and Grounding Evaluation 

In [63]:
import pandas as pd

EVAL_FILE = PROJECT_ROOT / "data" / "raw" / "policy_eval.csv"

eval_df = pd.read_csv(EVAL_FILE)

print("Evaluation questions:", len(eval_df))
display(eval_df.head())

Evaluation questions: 14


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more
